# Lab · Hồi quy tuyến tính 1D: dự đoán · LỜI GIẢI

> Chỉ mở file này khi bạn đã thử bài tập, hoặc đã dùng hết cả 3 gợi ý.

Bài tập cho [`lesson.md`](../lesson.md) (Module 3, Bài 1). Theo lab "Linear Regression 1D: Prediction" của khóa: dự đoán bằng tensor, bằng `nn.Linear`, bằng `nn.Module` tự viết; thêm phần làm quen hàm loss.

**Cách làm:** tự thử trước. Kẹt thì mở **Gợi ý 1**, rồi **Gợi ý 2**, cuối cùng mới **Gợi ý 3**. Lời giải ở thư mục `solution/`.

Môi trường do **uv** quản lý: trong thư mục khóa `04-intro-neural-networks-pytorch/`, chạy `uv sync` (một lần), rồi `uv run jupyter lab`.

## Chuẩn bị

In [ ]:
import torch
from torch import nn
from torch.nn import Linear

print("PyTorch", torch.__version__)

## Bài 1. Dự đoán bằng tensor

Mô hình `ŷ = -1 + 2x`. Tạo `w = 2.0` và `b = -1.0` (cả hai **có theo dõi gradient**), viết hàm `forward(x)` trả về `w * x + b`.
Dự đoán tại `x = [[1.0]]` và lưu vào `yhat1`. **Đoán trước** kết quả.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(-1.0, requires_grad=True)

def forward(x):
    return w * x + b

yhat1 = forward(torch.tensor([[1.0]]))
print(yhat1)

In [ ]:
assert w.requires_grad and b.requires_grad
assert yhat1.tolist() == [[1.0]], "2·1 − 1 = 1"
print("✅ Bài 1 đúng")

## Bài 2. Dự đoán nhiều mẫu một lúc

Dùng `x = torch.tensor([[1.0], [2.0]])`. Lưu `shape_x = x.shape` và `yhat2 = forward(x)`.
Câu hỏi: shape của `yhat2` là gì? Mỗi hàng của `x` là gì?

In [ ]:
x = torch.tensor([[1.0], [2.0]])
shape_x = x.shape
yhat2 = forward(x)
print(shape_x, yhat2)

In [ ]:
assert tuple(shape_x) == (2, 1)
assert yhat2.tolist() == [[1.0], [3.0]]
assert tuple(yhat2.shape) == (2, 1)
print("✅ Bài 2 đúng")

## Bài 3. Luyện tập: ba mẫu

Dự đoán cho `x = [[1.0], [2.0], [3.0]]` với `w`, `b` ở trên. Lưu vào `yhat3`.

In [ ]:
yhat3 = forward(torch.tensor([[1.0], [2.0], [3.0]]))
print(yhat3)

In [ ]:
assert yhat3.tolist() == [[1.0], [3.0], [5.0]]
print("✅ Bài 3 đúng")

## Bài 4. Lớp `nn.Linear`

Đặt `torch.manual_seed(1)`, rồi tạo `model = Linear(in_features=1, out_features=1, bias=True)`.
Lưu `w_val = model.weight.item()`, `b_val = model.bias.item()` và `keys = list(model.state_dict().keys())`.
Với seed 1, tham số khởi tạo là `w ≈ 0.5153`, `b ≈ -0.4414`.

In [ ]:
torch.manual_seed(1)
model = Linear(in_features=1, out_features=1, bias=True)
w_val = model.weight.item()
b_val = model.bias.item()
keys = list(model.state_dict().keys())
print(w_val, b_val, keys)

In [ ]:
assert abs(w_val - 0.5153) < 1e-3 and abs(b_val + 0.4414) < 1e-3
assert keys == ["weight", "bias"]
assert tuple(model.weight.shape) == (1, 1) and tuple(model.bias.shape) == (1,)
print("✅ Bài 4 đúng")

## Bài 5. Dự đoán bằng `model(x)`

Dùng `model` ở bài 4 để dự đoán cho `x = [[1.0], [2.0], [3.0]]`, lưu vào `yhat_model`.
Rồi tự kiểm tra: `manual` = `w * x + b` tính từ chính `model.weight` và `model.bias`. Hai kết quả có bằng nhau không? Lưu `same` (`True/False`).

In [ ]:
x = torch.tensor([[1.0], [2.0], [3.0]])
yhat_model = model(x)
manual = model.weight * x + model.bias
same = torch.allclose(yhat_model, manual)
print(yhat_model, same)

In [ ]:
assert tuple(yhat_model.shape) == (3, 1)
assert same is True
assert abs(yhat_model[0].item() - 0.0739) < 1e-3, "với seed 1, dự đoán tại x=1 khoảng 0.0739"
print("✅ Bài 5 đúng")

## Bài 6. Mô hình tự viết với `nn.Module`

Viết lớp `LR(nn.Module)`:
- `__init__(self, input_size, output_size)`: gọi `super().__init__()` rồi tạo `self.linear = nn.Linear(input_size, output_size)`;
- `forward(self, x)`: trả về `self.linear(x)`.

Tạo `lr_model = LR(1, 1)`. Lưu `keys2 = list(lr_model.state_dict().keys())`, và dự đoán `x = [[1.0], [2.0]]` vào `yhat_lr`.

In [ ]:
class LR(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.linear(x)

lr_model = LR(1, 1)
keys2 = list(lr_model.state_dict().keys())
yhat_lr = lr_model(torch.tensor([[1.0], [2.0]]))
print(keys2, yhat_lr)

In [ ]:
assert keys2 == ["linear.weight", "linear.bias"], "tên tham số có tiền tố tên thuộc tính"
assert tuple(yhat_lr.shape) == (2, 1)
print("✅ Bài 6 đúng")

## Bài 7. Luyện tập: tạo một mô hình `LR` mới

Tạo `lr1 = LR(1, 1)` (tham số mới, ngẫu nhiên) và dự đoán `x = [[1.0], [2.0], [3.0]]` vào `yhat_lr1`.
Hai mô hình `lr_model` và `lr1` có cùng tham số không? Lưu `different` (`True/False`).

In [ ]:
lr1 = LR(1, 1)
yhat_lr1 = lr1(torch.tensor([[1.0], [2.0], [3.0]]))
different = not torch.equal(lr_model.linear.weight, lr1.linear.weight)
print(yhat_lr1, different)

In [ ]:
assert tuple(yhat_lr1.shape) == (3, 1)
assert different is True
print("✅ Bài 7 đúng")

## Bài 8. Hàm loss MSE

`yhat = [1., 2., 3.]`, `y = [1., 1., 5.]`.
- `mse_manual`: tính MSE bằng công thức (trung bình của bình phương sai số), **không** dùng `nn.MSELoss`.
- `mse_torch`: tính bằng `nn.MSELoss()`.
Hai kết quả phải bằng nhau. Tính tay trước: `(0 + 1 + 4) / 3`.

In [ ]:
yhat = torch.tensor([1., 2., 3.])
y = torch.tensor([1., 1., 5.])
mse_manual = torch.mean((yhat - y) ** 2)
mse_torch = nn.MSELoss()(yhat, y)
print(mse_manual, mse_torch)

In [ ]:
assert abs(mse_manual.item() - 5 / 3) < 1e-6
assert torch.isclose(mse_manual, mse_torch)
print("✅ Bài 8 đúng")

## Bài 9. Loss là hàm của tham số

Dữ liệu `X = [[1.], [2.], [3.]]`, `Y = [[1.], [3.], [5.]]` (nằm đúng trên đường `y = 2x − 1`).
Cho các cặp tham số ứng viên `(w, b)`: `(2, -1)`, `(1, 0)`, `(0, 0)`, `(3, -2)`.
- `losses`: list MSE tương ứng với từng cặp (theo đúng thứ tự trên).
- `best`: cặp `(w, b)` cho loss nhỏ nhất.

In [ ]:
X = torch.tensor([[1.], [2.], [3.]])
Y = torch.tensor([[1.], [3.], [5.]])
candidates = [(2.0, -1.0), (1.0, 0.0), (0.0, 0.0), (3.0, -2.0)]
losses = []
for w_, b_ in candidates:
    yhat = w_ * X + b_
    losses.append(torch.mean((yhat - Y) ** 2).item())
best = candidates[int(torch.tensor(losses).argmin())]
print(losses, best)

In [ ]:
assert abs(losses[0]) < 1e-9, "đúng tham số thì loss = 0"
assert abs(losses[1] - 5 / 3) < 1e-5
assert best == (2.0, -1.0)
print("✅ Bài 9 đúng: loss nhỏ nhất đúng tại tham số thật")

## Hết